# 🌿 Plant Disease Detection using CNN
## Notebook 2 — Model Training & Evaluation (Phase 2)

### What this notebook does
1. Loads the same dataset split as Notebook 1.
2. Trains **Model A: a basic CNN built from scratch**.
3. Trains **Model B: MobileNetV2 with transfer learning** (pretrained on ImageNet, then fine-tuned).
4. Evaluates both on the hidden **test set**: accuracy, precision, recall, F1-score, confusion matrix.
5. Compares the two models.

### How to run
`Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. Total time on a free T4 GPU is roughly **45–75 minutes**. Keep the tab open.

> 💾 Models and results are saved to Google Drive (`MyDrive/plant-disease-project`). If Colab disconnects, run all again: **already-trained models are loaded instead of retrained** (set `RETRAIN = True` to force retraining).

In [ ]:
# ============================================================
# SETUP: imports, settings and project folders
# ============================================================
import os, sys, json, time, random, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42                     # fixed "random seed" -> same results every run
random.seed(SEED); np.random.seed(SEED)

IMG_SIZE = 224                # every image is resized to 224 x 224 pixels
BATCH_SIZE = 32               # how many images the model sees at once

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # Google Drive keeps our trained models & results safe even if Colab disconnects
    from google.colab import drive
    try:
        drive.mount('/content/drive')
        PROJECT_DIR = '/content/drive/MyDrive/plant-disease-project'
    except Exception as e:
        # Drive permission failed (tip: tick "Select all" on the Google permission screen).
        # We continue anyway, but results are only kept until this Colab session ends.
        print('WARNING: Google Drive could not be connected:', e)
        print('Saving to /content/outputs instead - download the results before closing Colab!')
        PROJECT_DIR = '/content/outputs'
    DATA_ROOT = '/content/data'              # fast local disk for the images
else:
    PROJECT_DIR = os.path.abspath('../outputs')
    DATA_ROOT = os.path.abspath('../data')

MODELS_DIR  = os.path.join(PROJECT_DIR, 'models')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')
FIG_DIR     = os.path.join(RESULTS_DIR, 'figures')
for d in [MODELS_DIR, RESULTS_DIR, FIG_DIR, DATA_ROOT]:
    os.makedirs(d, exist_ok=True)

DATASET_DIR = os.path.join(DATA_ROOT, 'PlantVillage-Dataset', 'raw', 'color')
print('Running in Colab:', IN_COLAB)
print('Results will be saved in:', PROJECT_DIR)

In [ ]:
RETRAIN = False          # True = train again even if a saved model exists
EPOCHS_CNN = 25          # max rounds for the basic CNN (early stopping usually ends sooner)
EPOCHS_TL_HEAD = 6       # MobileNetV2 stage 1: train only the new classifier head
EPOCHS_TL_FINE = 8       # MobileNetV2 stage 2: fine-tune the top of the pretrained network
USE_CLASS_WEIGHTS = True # give rare classes more importance during training

In [ ]:
# ============================================================
# DOWNLOAD THE PLANTVILLAGE DATASET (about 1 GB, takes ~2-5 minutes)
# Source: the official GitHub copy by S. P. Mohanty (author of our main reference paper)
# We download only the 'raw/color' folder = 54,305 original colour leaf photos.
# ============================================================
REPO_URL = 'https://github.com/spMohanty/PlantVillage-Dataset'
repo_dir = os.path.join(DATA_ROOT, 'PlantVillage-Dataset')

if os.path.isdir(DATASET_DIR) and len(os.listdir(DATASET_DIR)) == 38:
    print('Dataset already downloaded - skipping.')
else:
    t0 = time.time()
    subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse',
                    REPO_URL, repo_dir], check=True)
    subprocess.run(['git', '-C', repo_dir, 'sparse-checkout', 'set', 'raw/color'], check=True)
    print(f'Download finished in {time.time() - t0:.0f} seconds')

n_images = sum(len(files) for _, _, files in os.walk(DATASET_DIR))
print('Class folders:', len(os.listdir(DATASET_DIR)))
print('Total images :', n_images)     # should print 54305

In [ ]:
# ============================================================
# HELPER: turn folder names into readable names
# 'Tomato___Early_blight'  ->  'Tomato - Early blight'
# ============================================================
def pretty_name(folder_name):
    crop, disease = folder_name.split('___')
    crop = crop.replace('_', ' ').replace(',', '').strip()
    disease = disease.replace('_', ' ').strip()
    disease = disease[0].upper() + disease[1:]
    return f'{crop} - {disease}'

def crop_of(folder_name):
    return folder_name.split('___')[0].replace('_', ' ').replace(',', '').strip()

In [ ]:
# ============================================================
# BUILD A TABLE OF ALL IMAGES: one row per image -> (file path, class)
# ============================================================
IMG_EXTS = {'.jpg', '.jpeg', '.png'}
CLASS_NAMES = sorted(d.name for d in Path(DATASET_DIR).iterdir() if d.is_dir())
NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_ID = {c: i for i, c in enumerate(CLASS_NAMES)}

records = []
for c in CLASS_NAMES:
    for p in sorted((Path(DATASET_DIR) / c).iterdir()):
        if p.suffix.lower() in IMG_EXTS:
            # store the path RELATIVE to the dataset folder (works in any session)
            records.append((f'{c}/{p.name}', c, CLASS_TO_ID[c]))

df = pd.DataFrame(records, columns=['filepath', 'label', 'label_id'])
print(f'{len(df)} images in {NUM_CLASSES} classes')

# Save the list of class names - the app needs it later to turn numbers back into names
with open(os.path.join(MODELS_DIR, 'class_names.json'), 'w') as f:
    json.dump(CLASS_NAMES, f, indent=1)
df.head()

In [ ]:
# ============================================================
# SPLIT INTO TRAIN (70%) / VALIDATION (15%) / TEST (15%)
# 'stratify' keeps the same class proportions in every part,
# so rare classes (e.g. Potato healthy, 152 images) appear in all three.
# The split is saved, so every notebook uses EXACTLY the same images.
# ============================================================
from sklearn.model_selection import train_test_split

split_files = {s: os.path.join(RESULTS_DIR, f'split_{s}.csv') for s in ['train', 'val', 'test']}

if all(os.path.exists(f) for f in split_files.values()):
    train_df, val_df, test_df = (pd.read_csv(split_files[s]) for s in ['train', 'val', 'test'])
    print('Loaded the saved split.')
else:
    train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['label_id'],
                                         random_state=SEED)
    val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label_id'],
                                       random_state=SEED)
    for s, part in zip(['train', 'val', 'test'], [train_df, val_df, test_df]):
        part.reset_index(drop=True).to_csv(split_files[s], index=False)
    train_df, val_df, test_df = (pd.read_csv(split_files[s]) for s in ['train', 'val', 'test'])
    print('Created and saved a new split.')

print(f'Train: {len(train_df):6d} images')
print(f'Val  : {len(val_df):6d} images')
print(f'Test : {len(test_df):6d} images')

## Step 1 — Data pipeline
Same preprocessing as Notebook 1: read → decode → resize to 224×224 → batches of 32.

In [ ]:
# ============================================================
# PREPROCESSING PIPELINE (tf.data)
# For every image:  read file -> decode JPEG -> resize to 224x224
# Normalisation (0-255 -> 0-1) is done INSIDE the model as its first layer,
# so the saved model can be given a raw photo directly (this keeps the app simple).
# ============================================================
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])      # float32, values 0-255
    return img, label

def make_dataset(frame, shuffle=False):
    paths = [os.path.join(DATASET_DIR, p) for p in frame['filepath']]
    ds = tf.data.Dataset.from_tensor_slices((paths, frame['label_id'].values))
    if shuffle:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_df, shuffle=True)
val_ds   = make_dataset(val_df)
test_ds  = make_dataset(test_df)       # NOT shuffled -> predictions stay in the same order as test_df

images, labels = next(iter(train_ds))
print('One batch of images:', images.shape, images.dtype)
print('One batch of labels:', labels.shape)

In [ ]:
# ============================================================
# DATA AUGMENTATION: random flips, rotations, zooms and contrast changes.
# Only active during TRAINING - it turns 1 photo into many slightly different ones,
# so the model learns "what a disease looks like", not "where the leaf sits in the photo".
# ============================================================
def make_augmentation():
    return keras.Sequential([
        layers.RandomFlip('horizontal_and_vertical'),
        layers.RandomRotation(0.15),     # up to +/- 54 degrees
        layers.RandomZoom(0.15),         # zoom in/out up to 15%
        layers.RandomContrast(0.15),
    ], name='data_augmentation')

## Step 2 — Class weights (handling imbalance)
Some classes have ~3,800 training images, others only ~100. Without help, the model would focus on the big classes. **Class weights** make a mistake on a rare class "cost" more during training:

`weight(class) = total_images / (number_of_classes × images_in_that_class)`

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

weights = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=train_df['label_id'].values)
class_weight = {i: float(w) for i, w in enumerate(weights)} if USE_CLASS_WEIGHTS else None

cw = pd.Series(weights, index=[pretty_name(c) for c in CLASS_NAMES]).sort_values()
print('Lowest weights (big classes):'); print(cw.head(3).round(2).to_string())
print('\nHighest weights (rare classes):'); print(cw.tail(3).round(2).to_string())

## Step 3 — Helper functions for training and plotting

In [ ]:
def get_callbacks(name):
    return [
        # stop when validation loss hasn't improved for 5 rounds, and keep the best weights
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
        # halve the learning rate when progress stalls
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=1),
        # save the best model so far to Google Drive
        keras.callbacks.ModelCheckpoint(os.path.join(MODELS_DIR, f'{name}.keras'),
                                        monitor='val_loss', save_best_only=True, verbose=0),
    ]

def save_history(name, history_dict, train_seconds):
    with open(os.path.join(RESULTS_DIR, f'{name}_history.json'), 'w') as f:
        json.dump({'history': history_dict, 'train_seconds': train_seconds}, f)

def load_history(name):
    with open(os.path.join(RESULTS_DIR, f'{name}_history.json')) as f:
        d = json.load(f)
    return d['history'], d['train_seconds']

In [ ]:
def plot_history(history_dict, title, filename, fine_tune_start=None):
    """Accuracy and loss curves for training vs validation."""
    epochs = np.arange(1, len(history_dict['accuracy']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    for ax, metric, label in [(axes[0], 'accuracy', 'Accuracy'), (axes[1], 'loss', 'Loss')]:
        ax.plot(epochs, history_dict[metric], 'o-', label=f'Training {label.lower()}')
        ax.plot(epochs, history_dict['val_' + metric], 's-', label=f'Validation {label.lower()}')
        if fine_tune_start:
            ax.axvline(fine_tune_start + 0.5, color='gray', ls='--', label='Fine-tuning starts')
        ax.set_xlabel('Epoch'); ax.set_ylabel(label); ax.set_title(f'{title}: {label}')
        ax.grid(alpha=0.3); ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, filename), dpi=150)
    plt.show()

## Step 4 — Model A: Basic CNN (built from scratch)

| Layer group | What it does |
|---|---|
| Augmentation | random flip / rotate / zoom / contrast (training only) |
| Rescaling | pixels 0–255 → 0–1 (normalisation) |
| 5 × [Conv2D → BatchNorm → ReLU → MaxPool] | learns features: edges → textures → spots → disease patterns. Filters: 32, 64, 128, 256, 256 |
| GlobalAveragePooling | summarises each feature map into one number |
| Dropout 0.4 → Dense 256 → Dropout 0.3 | combines features; dropout prevents memorising |
| Dense 38 + Softmax | one probability per class (they add up to 1) |

In [ ]:
def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)   # 3x3 filters scan the image
    x = layers.BatchNormalization()(x)                                 # keeps values stable -> faster training
    x = layers.Activation('relu')(x)                                   # keeps only positive signals
    x = layers.MaxPooling2D()(x)                                       # halves width & height
    return x

def build_basic_cnn(num_classes):
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = make_augmentation()(inputs)
    x = layers.Rescaling(1. / 255)(x)
    for filters in [32, 64, 128, 256, 256]:
        x = conv_block(x, filters)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return keras.Model(inputs, outputs, name='basic_cnn')

cnn_path = os.path.join(MODELS_DIR, 'basic_cnn.keras')
cnn_hist_path = os.path.join(RESULTS_DIR, 'basic_cnn_history.json')

In [ ]:
tf.keras.utils.set_random_seed(SEED)

if os.path.exists(cnn_path) and os.path.exists(cnn_hist_path) and not RETRAIN:
    print('Loading the already-trained basic CNN from Drive...')
    cnn_model = keras.models.load_model(cnn_path)
    cnn_history, cnn_train_seconds = load_history('basic_cnn')
else:
    cnn_model = build_basic_cnn(NUM_CLASSES)
    cnn_model.compile(optimizer=keras.optimizers.Adam(1e-3),
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])
    cnn_model.summary()
    t0 = time.time()
    h = cnn_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_CNN,
                      class_weight=class_weight, callbacks=get_callbacks('basic_cnn'))
    cnn_train_seconds = time.time() - t0
    cnn_history = {k: [float(v) for v in vals] for k, vals in h.history.items()}
    cnn_model.save(cnn_path)                # best weights (restored by EarlyStopping)
    save_history('basic_cnn', cnn_history, cnn_train_seconds)

print(f'Training time: {cnn_train_seconds/60:.1f} min  |  parameters: {cnn_model.count_params():,}')

In [ ]:
plot_history(cnn_history, 'Basic CNN', 'basic_cnn_training_curves.png')

## Step 5 — Model B: MobileNetV2 (transfer learning)
MobileNetV2 has already learned general visual features (edges, textures, shapes) from **1.4 million ImageNet photos**. We reuse that knowledge:

- **Stage 1:** freeze the pretrained network and train only a new classifier head for our 38 classes.
- **Stage 2 (fine-tuning):** unfreeze the top 40 layers and train them with a **very small learning rate** so they adapt to leaf diseases without forgetting what they know.

MobileNetV2 expects pixels in the range −1 to 1, so its normalisation layer is `Rescaling(1/127.5, offset=-1)`.

In [ ]:
def build_mobilenetv2(num_classes):
    base = keras.applications.MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3),
                                          include_top=False, weights='imagenet')
    base.trainable = False                          # Stage 1: freeze pretrained layers
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = make_augmentation()(inputs)
    x = layers.Rescaling(1. / 127.5, offset=-1)(x)  # 0-255 -> -1..1
    x = base(x, training=False)                     # keep BatchNorm layers in inference mode
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return keras.Model(inputs, outputs, name='mobilenetv2_transfer'), base

tl_path = os.path.join(MODELS_DIR, 'mobilenetv2.keras')
tl_hist_path = os.path.join(RESULTS_DIR, 'mobilenetv2_history.json')

In [ ]:
tf.keras.utils.set_random_seed(SEED)

if os.path.exists(tl_path) and os.path.exists(tl_hist_path) and not RETRAIN:
    print('Loading the already-trained MobileNetV2 from Drive...')
    tl_model = keras.models.load_model(tl_path)
    tl_history, tl_train_seconds = load_history('mobilenetv2')
else:
    tl_model, base = build_mobilenetv2(NUM_CLASSES)

    # ---------- Stage 1: train the new head only ----------
    tl_model.compile(optimizer=keras.optimizers.Adam(1e-3),
                     loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    tl_model.summary()
    t0 = time.time()
    h1 = tl_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_TL_HEAD,
                      class_weight=class_weight, callbacks=get_callbacks('mobilenetv2'))

    # ---------- Stage 2: fine-tune the top 40 layers ----------
    base.trainable = True
    for layer in base.layers[:-40]:
        layer.trainable = False
    for layer in base.layers:                       # BatchNorm layers stay frozen (standard practice)
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    tl_model.compile(optimizer=keras.optimizers.Adam(1e-5),     # 100x smaller learning rate
                     loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    stage1_epochs = len(h1.history['loss'])
    h2 = tl_model.fit(train_ds, validation_data=val_ds,
                      epochs=stage1_epochs + EPOCHS_TL_FINE, initial_epoch=stage1_epochs,
                      class_weight=class_weight, callbacks=get_callbacks('mobilenetv2'))
    tl_train_seconds = time.time() - t0

    tl_history = {k: [float(v) for v in h1.history[k]] + [float(v) for v in h2.history.get(k, [])]
                  for k in ['accuracy', 'loss', 'val_accuracy', 'val_loss']}
    tl_history['stage1_epochs'] = stage1_epochs
    tl_model.save(tl_path)
    save_history('mobilenetv2', tl_history, tl_train_seconds)

print(f'Training time: {tl_train_seconds/60:.1f} min  |  parameters: {tl_model.count_params():,}')

In [ ]:
plot_history(tl_history, 'MobileNetV2 (transfer learning)', 'mobilenetv2_training_curves.png',
             fine_tune_start=tl_history.get('stage1_epochs'))

## Step 6 — Final evaluation on the hidden TEST set
Now, for the first time, both models see the **test images**.

| Metric | Simple meaning |
|---|---|
| **Accuracy** | % of all test images classified correctly |
| **Precision** | When the model says "Late blight", how often is it right? |
| **Recall** | Of all real "Late blight" leaves, how many did the model find? |
| **F1-score** | Balance of precision and recall (high only if both are high) |
| **Macro average** | Average over classes, so every class counts equally, even small ones |
| **Top-3 accuracy** | Is the correct answer among the model's 3 best guesses? |
| **Confusion matrix** | Grid showing which classes get confused with which |

In [ ]:
y_true = test_df['label_id'].values
cnn_probs = cnn_model.predict(test_ds, verbose=1)
tl_probs  = tl_model.predict(test_ds, verbose=1)
np.save(os.path.join(RESULTS_DIR, 'basic_cnn_test_probs.npy'), cnn_probs)
np.save(os.path.join(RESULTS_DIR, 'mobilenetv2_test_probs.npy'), tl_probs)

In [ ]:
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, classification_report,
                             confusion_matrix, top_k_accuracy_score)

try:
    from IPython.display import display
except ImportError:
    display = print

SHORT = [pretty_name(c) for c in CLASS_NAMES]

def evaluate_model(name, display_name, probs, y_true):
    y_pred = probs.argmax(axis=1)
    acc = accuracy_score(y_true, y_pred)
    p_m, r_m, f_m, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    p_w, r_w, f_w, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
    top3 = top_k_accuracy_score(y_true, probs, k=3, labels=np.arange(NUM_CLASSES))

    print(f'================  {display_name}  ================')
    print(f'Test accuracy       : {acc*100:.2f}%')
    print(f'Top-3 accuracy      : {top3*100:.2f}%')
    print(f'Macro precision     : {p_m*100:.2f}%')
    print(f'Macro recall        : {r_m*100:.2f}%')
    print(f'Macro F1-score      : {f_m*100:.2f}%')
    print(f'Weighted F1-score   : {f_w*100:.2f}%')

    # ---- per-class report ----
    report = classification_report(y_true, y_pred, labels=np.arange(NUM_CLASSES), target_names=SHORT,
                                   digits=4, output_dict=True, zero_division=0)
    report_df = pd.DataFrame(report).T
    report_df.to_csv(os.path.join(RESULTS_DIR, f'{name}_classification_report.csv'))

    # ---- confusion matrix (each row shows where the images of that class went, in %) ----
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
    cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    plt.figure(figsize=(20, 17))
    sns.heatmap(cm_norm * 100, cmap='Greens', xticklabels=SHORT, yticklabels=SHORT,
                annot=cm_norm.shape[0] <= 12, fmt='.0f', cbar_kws={'label': '% of true class'},
                vmin=0, vmax=100, linewidths=0.2, linecolor='#eeeeee')
    plt.xlabel('Predicted class'); plt.ylabel('True class')
    plt.title(f'{display_name}: normalised confusion matrix (test set)')
    plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, f'{name}_confusion_matrix.png'), dpi=130); plt.show()

    # ---- the most frequent mistakes ----
    off = cm.copy(); np.fill_diagonal(off, 0)
    pairs = [(SHORT[i], SHORT[j], int(off[i, j])) for i, j in zip(*np.unravel_index(np.argsort(off, axis=None)[::-1], off.shape))
             if off[i, j] > 0][:10]
    mistakes = pd.DataFrame(pairs, columns=['True class', 'Predicted as', 'Count'])
    print('\nMost common confusions:')
    display(mistakes)

    return {'Model': display_name, 'Test accuracy (%)': acc * 100, 'Top-3 accuracy (%)': top3 * 100,
            'Macro precision (%)': p_m * 100, 'Macro recall (%)': r_m * 100, 'Macro F1 (%)': f_m * 100,
            'Weighted F1 (%)': f_w * 100}, report_df, y_pred

In [ ]:
cnn_summary, cnn_report, cnn_pred = evaluate_model('basic_cnn', 'Basic CNN', cnn_probs, y_true)

In [ ]:
tl_summary, tl_report, tl_pred = evaluate_model('mobilenetv2', 'MobileNetV2 (transfer learning)', tl_probs, y_true)

## Step 7 — Model comparison

In [ ]:
def model_size_mb(path):
    return os.path.getsize(path) / 1e6 if os.path.exists(path) else float('nan')

cnn_summary.update({'Parameters': cnn_model.count_params(), 'Model size (MB)': model_size_mb(cnn_path),
                    'Training time (min)': cnn_train_seconds / 60})
tl_summary.update({'Parameters': tl_model.count_params(), 'Model size (MB)': model_size_mb(tl_path),
                   'Training time (min)': tl_train_seconds / 60})
comparison = pd.DataFrame([cnn_summary, tl_summary]).set_index('Model')
comparison.to_csv(os.path.join(RESULTS_DIR, 'model_comparison.csv'))
comparison.round(2)

In [ ]:
# Per-class F1-score for both models: shows which diseases are hard
f1 = pd.DataFrame({'Basic CNN': cnn_report.loc[SHORT, 'f1-score'],
                   'MobileNetV2': tl_report.loc[SHORT, 'f1-score']}).sort_values('MobileNetV2')
f1.plot(kind='barh', figsize=(10, 12), width=0.8)
plt.xlabel('F1-score'); plt.xlim(max(0, f1.values.min() - 0.05), 1.0)
plt.title('Per-class F1-score on the test set'); plt.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, 'per_class_f1_comparison.png'), dpi=140); plt.show()

print('5 hardest classes for MobileNetV2:')
print(f1['MobileNetV2'].head(5).round(4).to_string())

## Step 8 — Example predictions
Green title = correct, red title = wrong. Each title shows the prediction, its confidence, and the true class.

In [ ]:
from PIL import Image

def show_predictions(probs, y_pred, title, filename, n=12):
    rng = np.random.default_rng(SEED)
    wrong = np.where(y_pred != y_true)[0]
    right = np.where(y_pred == y_true)[0]
    # show mostly correct ones plus up to 4 mistakes, so we learn from errors too
    n_wrong = min(4, len(wrong))
    idx = np.concatenate([rng.choice(right, min(n - n_wrong, len(right)), replace=False),
                          rng.choice(wrong, n_wrong, replace=False) if n_wrong else np.array([], int)]).astype(int)
    plt.figure(figsize=(16, 11))
    for k, i in enumerate(idx):
        img = Image.open(os.path.join(DATASET_DIR, test_df['filepath'].iloc[i]))
        ok = y_pred[i] == y_true[i]
        plt.subplot(3, 4, k + 1); plt.imshow(img); plt.axis('off')
        plt.title(f'Pred: {SHORT[y_pred[i]]} ({probs[i].max()*100:.1f}%)\nTrue: {SHORT[y_true[i]]}',
                  color='green' if ok else 'red', fontsize=8.5)
    plt.suptitle(title); plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, filename), dpi=120); plt.show()

show_predictions(tl_probs, tl_pred, 'MobileNetV2: example test predictions', 'mobilenetv2_example_predictions.png')
show_predictions(cnn_probs, cnn_pred, 'Basic CNN: example test predictions', 'basic_cnn_example_predictions.png')

In [ ]:
# Save a few test images to Drive as examples for the web app (Notebook 3)
import shutil
EX_DIR = os.path.join(PROJECT_DIR, 'examples'); os.makedirs(EX_DIR, exist_ok=True)
wanted = ['Tomato___healthy', 'Tomato___Early_blight', 'Tomato___Late_blight',
          'Potato___Late_blight', 'Apple___Apple_scab', 'Grape___Black_rot']
for c in wanted:
    rows = test_df[test_df['label'] == c]
    if len(rows):
        src = os.path.join(DATASET_DIR, rows['filepath'].iloc[0])
        shutil.copy(src, os.path.join(EX_DIR, f'{c}.jpg'))
print('Example images saved:', sorted(os.listdir(EX_DIR)))

## ✅ How to read your results (for the report and viva)
- **Training curves:** training and validation accuracy should rise together. If training accuracy keeps rising while validation accuracy falls, the model is **overfitting**. Early stopping and dropout prevent this.
- **Transfer learning vs scratch CNN:** MobileNetV2 usually reaches higher accuracy in less training time because it starts with knowledge from ImageNet. Most of its layers are frozen in stage 1, so each epoch is also faster. It is still small (~2.3M parameters) and designed for mobile phones.
- **Confusion matrix:** nearly all values should sit on the diagonal. Off-diagonal cells show real confusions; these are usually between diseases of the **same crop** with similar spots (e.g. tomato Early blight vs Target spot), which matches the challenge stated in our problem statement.
- **Limitation:** PlantVillage photos are taken in a lab (single leaf, plain background), so very high test accuracy does **not** guarantee the same accuracy on farm photos. Notebook 3 tests this on real field images (PlantDoc).

All numbers above are saved in `MyDrive/plant-disease-project/results/` and all figures in `results/figures/`.

**Next:** `03_web_app_and_field_test.ipynb`